## Part 0 — Rebuild Lab 1 data and pipeline
Since lab 2 is a continuation from Lab 1 we have to get the Lab 1  data here for lab 2

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Load the data (same file as Lab 1)
csv_path = Path("../data/housing.csv")
housing = pd.read_csv(csv_path)

# 2. Income bands, used only for the stratified split
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

# 3. Stratified split - same seed, so same rows as Lab 1
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

# 4. Remove income_cat - it was only needed for splitting
strat_train_set = strat_train_set.drop("income_cat", axis=1)
strat_test_set = strat_test_set.drop("income_cat", axis=1)

# 5. Rebuild the preprocessing pipeline
num_attribs = [
    "longitude", "latitude", "housing_median_age", "total_rooms",
    "total_bedrooms", "population", "households", "median_income"
]
cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs)
])

print("Train set:", strat_train_set.shape)
print("Test set:", strat_test_set.shape)

Train set: (16512, 10)
Test set: (4128, 10)


## Part I — Preparing the Data for Model Training

In [2]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [3]:
housing_prepared = full_pipeline.fit_transform(housing)

In [4]:
housing_prepared.shape

(16512, 13)

# Task 1.1 – Understand the Prepared Data


**1. What is the target variable?**
**Answer:** The target value is the `median_house_value`.

**2. What are the input features?**
**Answer:** There are 9 input features. These are the other columns apart from the `median_house_value`. These are `longitude`, `latitude`, `total_rooms`, `total_bedrooms`, `population`, `households`, `median_income`, `ocean_proximity`, `housing_median_age`. 

**3. How many training observations are available?**
**Answer:**  The training observation is 80% of the total district. That is 80% of 20640 which is 16512.

**4. How many features are available after preprocessing?**
**Answer:** There are 13 features after preprocessing.

**5. Why can the number of features be different from the original number of attributes?**
**Answer:** The number of features can change if we have a feature that is a nominal category and is One-Hot Encoded. this will increase the number of columns. For example after One-Hot Encoding in our housing dataset, we get 5 extra columns that replaces the One-Hot encoded category column (`ocean_proximity`). Therefor 7 exisitng numerical columns + the new columns replacing the `ocean_proximity` = 13 columns.

## Part II — Train a Linear Regression Model

In [5]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](13,)","[-53381.26,-54311.56, 13122.77,...,198442.4 ,-43374.75,-35994.43]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2.587e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(12)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](13,)","[254.16,180.92,133.77,..., 17.92, 1.58, 0. ]"


In [6]:
print("Number of weights:", lin_reg.coef_.shape)
print("Intercept (b):", lin_reg.intercept_)
print("Weights (w):", lin_reg.coef_.round(0))

Number of weights: (13,)
Intercept (b): 258682.58180082325
Weights (w): [-53381. -54312.  13123.  -8357.  26159. -50242.  37563.  73068. -39694.
 -79380. 198442. -43375. -35994.]


# Task 2.1 – Train the Model

Train the Linear Regression model and confirm that the model has been fitted successfully. Consider the following questions:

**1. What does the fit() operation do?**
**Answer:** The `fit()`operation simply enables the model to make predictions. without fitting, the model cannot predict. The model gets the inputs and the correct prices together, learns from it and then use learned patterns to make predictions. It adjusts its weights until its guesses are as close as possible to the real prices.

**2. What information does the model learn from the training data?**
**Answer:** The model learns the 13 weights (from the columns) and 1 learned (base) intercept. Each feature weight shows how each column pushes the price of the houses up or down.

**3. Does the model “memorize” the house prices?**
**Answer:** No, the model does not memorize. It instead learns patterns from the training data and then use to understand the dataset more and even predict unknown and new data. Related to our concerned dataset, it learns the 13 weights and 1 intercept (base) and then use that knowlegde for test data and unknown real life data.

## Part III — Generate and Interpret Predictions

In [7]:
some_data = housing.iloc[:5]
some_labels = housing_labels.iloc[:5]

some_data_prepared = full_pipeline.transform(some_data)

predictions = lin_reg.predict(some_data_prepared)

print("Predictions:", predictions)
print("Actual values:", list(some_labels))

Predictions: [270886.93228264 334830.60618945 119856.18008898 109210.92995275
 305575.65811433]
Actual values: [458300.0, 483800.0, 101700.0, 96100.0, 361800.0]


In [8]:
comparison = pd.DataFrame({
    "Actual": list(some_labels),
    "Predicted": predictions.round(0),
})
comparison["Error"] = comparison["Actual"] - comparison["Predicted"]
comparison

,Actual,Predicted,Error
0,458300.0,270887.0,187413.0
1,483800.0,334831.0,148969.0
2,101700.0,119856.0,-18156.0
3,96100.0,109211.0,-13111.0
4,361800.0,305576.0,56224.0


# Task 3.1 – Interpret the Predictions

See above small table comparing: actual house value, predicted house value, and prediction error.
Discuss:

**1. Are the predictions close to the actual values?**
**Answer:** Not really. the closest from the output table are row 2 and 3 with the error range 13000-18000.

**2. Are some predictions substantially different?**
**Answer:** Yes! Rows 0 and 1 shows a predicted price too far from the actual prices. Off by roughly 149,000–187,000. Both are expensive districts, and the model guessed too low

**3. What could cause the prediction errors?**
**Answer:** Here are some reasons that cause these prediction erros: The model is too simple and unerfitting. A linear rule cant follow complicated patterns and then secondly, it misses otehr important information liek crimes, hosue quality, school etc which can help make its prediction better.

## Part IV — Evaluate the Linear Regression Model

In [9]:
from sklearn.metrics import mean_squared_error
import numpy as np

housing_predictions = lin_reg.predict(housing_prepared)

lin_rmse = np.sqrt(
    mean_squared_error(housing_labels, housing_predictions)
)
print("RMSE:", lin_rmse)

RMSE: 68232.83515124217


In [10]:
housing_labels.describe()

count     16512.000000
mean     206333.518653
std      115314.047529
min       14999.000000
25%      119200.000000
50%      179200.000000
75%      263925.000000
max      500001.000000
Name: median_house_value, dtype: float64

# Task 4.1 – Interpret RMSE

Answer:

**1. What does RMSE measure?**
**Answer:** Root Mean Square Error measures how far a model's prediction is from the actual values. It gives the typical size of the error as one number, and big errors count more (because of the squaring).

**2. What are the units of RMSE in this problem?**
**Answer:** The units are Dollars. The same unit for `median_house_value`. Squaring gives dollars², and the root brings it back to dollars.

**3. Is a lower RMSE better or worse?**
**Answer:** A lower Root Mean Square Error is better. The closer it is to 0, the better.

**4. Relate the RMSE value to the typical house prices in the dataset**
**Answer:** With an RMSE of 68232.8 and a typical house price value (taking the median house price) of 179200 Dollars. Doing the maths thats approx 38% error which is a huge error margin. The error was measured on the training data, which is data the model already saw while learning. Even on familiar data, it's still off by about 38%. So the model is too simple → underfitting. For example, A student scores badly even on the practice questions they studied. That tells you the student didn't learn well enough.

## Part V — Train a Decision Tree Regressor

In [11]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with

In [12]:
housing_predictions = tree_reg.predict(housing_prepared)

tree_rmse = np.sqrt(
    mean_squared_error(housing_labels, housing_predictions)
)
print("Decision Tree RMSE:", tree_rmse)

Decision Tree RMSE: 0.0


# Task 5.1 – Compare Linear Regression and Decision Tree

Compare the RMSE of: Linear Regression vs Decision Tree Regression
Answer:

**1. Which model has the lower training RMSE?**
**Answer:** The decision tree model has an RMSE of 0.0, therefore it is lower than that of linear regression of 68232.8.

**2. Does a lower training error necessarily mean that the model will perform better on unseen data?**
**Answer:** No. Training error means the error is from data which it knows already. This says nothing about data that it doesnt know anything about. Using the analogy of a student who memorizes an exam questions and scores a perfect 100% because he has seen the questions before the exam. Give the student a different set of questions he has not seen before and they are more likely to fail.

**3. What might happen if the Decision Tree is too complex?**
**Answer:** If the decision tree is too complex, the model will learn everything including the noise (overfitting). More like memorizing everything. Hence it does bad on new data.

## Part VI — Investigate Overfitting

In [13]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    tree_reg,
    housing_prepared,
    housing_labels,
    scoring="neg_mean_squared_error",
    cv=10
)

tree_rmse_scores = np.sqrt(-scores)

In [14]:
print("Mean RMSE:", tree_rmse_scores.mean())
print("Standard deviation:", tree_rmse_scores.std())

Mean RMSE: 68730.84229824627
Standard deviation: 1585.362175240545


In [15]:
print("All 10 scores:", tree_rmse_scores.round(0))

All 10 scores: [69057. 67221. 67397. 70029. 68459. 71464. 71044. 68689. 67301. 66648.]


# Task 6.1 – Interpret Cross-Validation

Answer:

**1. Why do we use multiple validation folds?**
**Answer:** We use multiple validation folds to get a fair result. It could be a lucky guess if we test on one fold, but on multiple fold, it makes the result more reliable. For example judging how good a striker is on a span of one match is less realistic than judging over 10 matches. After 10 matches, one can judge how good the striker is.

**2. Why is the score returned by Scikit-learn negative?**
**Answer:** This is because Scikit-learn naturally assumes a bigger number to be better, but when dealing with errors, a smaller number is better. Therefore Scikit-learn adds the negative signs to even this out. ie: 'biggest number wins' rule still picks the smallest error".

**3. What does the mean RMSE tell you?**
**Answer:** The mean RMSE gotten is 68730.8 ~ $68731. That figure is the typical error, in dollars, on data the model didn't see. Using the median hosue price of 179200, 68731/179200 = 0.38 ~ 38% which is still a big error margin. The 10 scores are close together, so the model's error is about the same in every round.

**4. What does the standard deviation tell you?**
**Answer:** The standard deviation is 1585. It's supposed to show us how far the 10 RMSE scores are from their mean. So using the mean of 68731, 1585/68731 ≈ 0.023 ≈ 2.3% across the 10 folds. It shows consistency.

**5. Why can cross-validation provide a more useful estimate than a single training score?**
**Answer:** Cross validation provides a more useful estimate because it reveals overfitting, while the training score hides it. the Decision Tree's training RMSE was 0, and its cross-validation RMSE was 68,731. This proves it.

In [16]:
## Part VII — Compare Different Models Using Cross-Validation

In [17]:
lin_scores = cross_val_score(
    lin_reg,
    housing_prepared,
    housing_labels,
    scoring="neg_mean_squared_error",
    cv=10
)

lin_rmse_scores = np.sqrt(-lin_scores)

print("Mean RMSE:", lin_rmse_scores.mean())
print("Standard deviation:", lin_rmse_scores.std())

Mean RMSE: 68316.17692806528
Standard deviation: 971.5373052470952


In [18]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

forest_scores = cross_val_score(
    forest_reg,
    housing_prepared,
    housing_labels,
    scoring="neg_mean_squared_error",
    cv=10
)

forest_rmse_scores = np.sqrt(-forest_scores)

print("Mean RMSE:", forest_rmse_scores.mean())
print("Standard deviation:", forest_rmse_scores.std())

Mean RMSE: 48901.41320543065
Standard deviation: 670.5367015311969


# Task 7 – Compare the Models

| Model | Mean RMSE | Std. Dev | % Consistency Measure $ (Std ÷ Mean RMSE) | Mean RMSE as % of Median Price Median house price = $179,200 (from `describe()`)
|-------| ----------|----------| --------------------------------------| ------------------------------------------------ |
|Linear Regression | 68,316 | 972 | 1.4% | 38% |
|Decision Tree | 68,731 | 1585 | 2.3% | 38% |
|Random Forest | 48,901 | 671 | 1.4% | 27% |

Answer:

**1. Which model performs best according to the mean RMSE?**
**Answer:** The Random Forest model performs best both in terms of consistency and error. It has the lowest error.

**2. Which model gives the most consistent results?**
**Answer:** As mentioned above the Random Forest model gives the most consistent result with consistency percentage of 1.4%. Linear regression is also 1.4% but Random forest has the lowest Standard deviation.

**3. Is the model with the lowest training error also the best model according to cross-validation?**
**Answer:** Decision Tree has the lowest training error but it is not the best according to cross validation. Random forest is the best according to cross validation.

**4. What does this comparison tell you about evaluating machine learning models?**
**Answer:** Not to trust on trainihg data only. test on data that is unknown to the model using cross validation to measure.


## Part VIII — Fine-Tuning the Model

In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"n_estimators": [3, 10, 30], "max_features": [2, 4, 6, 8]},
    {"bootstrap": [False], "n_estimators": [3, 10], "max_features": [2, 3, 4]}
]

forest_reg = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    forest_reg,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)
grid_search.fit(housing_prepared, housing_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'max_features': [2, 4, ...], 'n_estimators': [3, 10, ...]}, {'bootstrap': [False], 'max_features': [2, 3, ...], 'n_estimators': [3, 10]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` w

In [21]:
print(grid_search.best_params_)
print(grid_search.best_estimator_)

{'max_features': 6, 'n_estimators': 30}
RandomForestRegressor(max_features=6, n_estimators=30, random_state=42)


In [22]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(round(np.sqrt(-mean_score)), params)

63905 {'max_features': 2, 'n_estimators': 3}
54821 {'max_features': 2, 'n_estimators': 10}
52262 {'max_features': 2, 'n_estimators': 30}
60176 {'max_features': 4, 'n_estimators': 3}
52739 {'max_features': 4, 'n_estimators': 10}
49928 {'max_features': 4, 'n_estimators': 30}
59071 {'max_features': 6, 'n_estimators': 3}
51808 {'max_features': 6, 'n_estimators': 10}
49323 {'max_features': 6, 'n_estimators': 30}
58271 {'max_features': 8, 'n_estimators': 3}
51553 {'max_features': 8, 'n_estimators': 10}
49405 {'max_features': 8, 'n_estimators': 30}
61434 {'bootstrap': False, 'max_features': 2, 'n_estimators': 3}
53034 {'bootstrap': False, 'max_features': 2, 'n_estimators': 10}
59373 {'bootstrap': False, 'max_features': 3, 'n_estimators': 3}
52843 {'bootstrap': False, 'max_features': 3, 'n_estimators': 10}
58956 {'bootstrap': False, 'max_features': 4, 'n_estimators': 3}
52094 {'bootstrap': False, 'max_features': 4, 'n_estimators': 10}


In [23]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    "n_estimators": randint(low=1, high=200),
    "max_features": randint(low=1, high=8)
}

forest_reg = RandomForestRegressor(random_state=42)

rnd_search = RandomizedSearchCV(
    forest_reg,
    param_distributions=param_distribs,
    n_iter=10,
    cv=5,
    scoring="neg_mean_squared_error",
    random_state=42,
)
rnd_search.fit(housing_prepared, housing_labels)

print(rnd_search.best_params_)
print("Best RMSE:", np.sqrt(-rnd_search.best_score_))

{'max_features': 7, 'n_estimators': 180}
Best RMSE: 48704.58467953158


# Task 8.1 – Investigate the Grid Search

Answer:

**1. Which combination of hyperparameters produced the best result?**
**Answer:** max_features = 6 and n_estimators = 30, with a cross-validation RMSE of about 49,323.

**2. How much did the performance vary between different combinations?**
**Answer:** A lot. The best combination scored about 49,323, and the worst (max_features = 2, n_estimators = 3) scored about 63,905. That's a difference of roughly 14,600. Combinations with only 3 trees were always the worst, and those with 30 trees were always the best. So the number of trees mattered most.

**3. Why is systematic hyperparameter search useful?**
**Answer:** Trying values by hand is slow, easy to get wrong, and you might miss the best combination. Grid search tests every combination automatically, scores each one fairly with cross-validation, and picks the best one. The same code always gives the same result, so it's repeatable.

# Task 8.2 – Compare Grid Search and Randomized Search

Discuss:

**1. What is the difference between Grid Search and Randomized Search?**
**Answer:** Grid Search tries every combination from fixed lists of values. Randomized Search tries a fixed number of random combinations (here 10), picked from ranges of values.

**2. Which method evaluates more combinations in this experiment?**
**Answer:** Grid Search: 18 combinations, versus 10 for Randomized Search.

**3. When could Randomized Search be preferable?**
**Answer:** When there are many hyperparameters or wide ranges of values. The number of grid combinations grows very fast. For example, 5 hyperparameters with 10 values each = 100,000 combinations. Randomized Search lets you control the time with n_iter. It can also try values you wouldn't have put in a list (like 180 trees).

**4. Did Randomized Search find a better model in your experiment?**
**Answer:** Yes, slightly. Its best was max_features = 7, n_estimators = 180, with an RMSE of about 48,705, compared with about 49,323 for Grid Search. That's about 600 better, even though it tested fewer combinations. But it used much bigger forests (180 trees vs 30), so it took longer to run.

## Part IX — Analyze the Best Model and Model Error

In [24]:
feature_importances = grid_search.best_estimator_.feature_importances_
print(feature_importances)

[1.19607145e-01 1.07532793e-01 4.47470104e-02 3.47843589e-02
 3.04676303e-02 4.41524812e-02 2.78180627e-02 4.04647812e-01
 1.39017605e-02 1.62404281e-01 2.88019085e-04 2.53503952e-03
 7.11360612e-03]


In [25]:
feature_names = full_pipeline.get_feature_names_out()

importance_table = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances.round(3)
}).sort_values("importance", ascending=False)

importance_table

,feature,importance
7,num__median_income,0.405
9,cat__ocean_proximity_INLAND,0.162
0,num__longitude,0.120
1,num__latitude,0.108
2,num__housing_median_age,0.045
5,num__population,0.044
3,num__total_rooms,0.035
4,num__total_bedrooms,0.030
6,num__households,0.028
8,cat__ocean_proximity_<1H OCEAN,0.014


# Task 9.1 – Interpret Feature Importance

Identify the most important features according to the selected Random Forest model.

Discuss:

**1. Which features appear most important?**
**Answer:** Median_income is by far the most important (about 0.41, or 41% of the model's decisions). After that come ocean_proximity_INLAND (about 0.16), longitude (about 0.12) and latitude (about 0.11). So income and location drive the predictions. The other features matter little, and ISLAND has almost no importance, because only a handful of districts are on islands.

**2. Does feature importance mean that a feature causes the target to change?**
**Answer:** No. Feature importance only shows which features the model used most to make good predictions. It does not prove cause and effect. For example, high income and high prices go together, but the model can't tell whether income causes prices, prices attract high earners, or something else (like a good location) drives both. "Useful for predicting" is not the same as "causes".

**3. Are the results consistent with the correlation analysis performed in Lab I?**
**Answer:** Mostly yes. In Lab I, median_income had the strongest correlation with house value, and here it's also the most important feature. One difference: location (longitude, latitude, INLAND) is much more important here than its correlations suggested. That's because correlation only measures straight-line relationships, while a Random Forest can learn complicated patterns. For example, "this specific area along the coast is expensive" can't be described by one straight line.

**4. How could feature importance help an engineer or domain expert?**

**Answer:** 
- It shows which data matters most, so effort can go into collecting that data accurately (income, location).
- It shows which features could be removed (like ISLAND), making the model simpler and faster.
- It lets experts check that the model makes sense. A real-estate expert would agree that income and location matter. If a strange feature came out on top, that would be a warning sign.
- It helps explain the model's decisions to non-technical people.

In [26]:
final_model = grid_search.best_estimator_

predictions = final_model.predict(housing_prepared)

errors = housing_labels - predictions

error_rmse = np.sqrt(np.mean(errors ** 2))
print("RMSE:", error_rmse)

RMSE: 19313.761356524363


In [27]:
from sklearn.model_selection import cross_val_predict

cv_predictions = cross_val_predict(
    RandomForestRegressor(max_features=6, n_estimators=30, random_state=42),
    housing_prepared,
    housing_labels,
    cv=5,
    n_jobs=-1
)

cv_errors = housing_labels - cv_predictions
print("Honest (CV) RMSE:", np.sqrt(np.mean(cv_errors ** 2)))

error_table = housing.copy()
error_table["actual"] = housing_labels
error_table["predicted"] = cv_predictions.round()
error_table["abs_error"] = cv_errors.abs().round()

print(error_table["abs_error"].describe())

Honest (CV) RMSE: 49323.24311753467
count     16512.00000
mean      32653.26617
std       36967.95053
min           0.00000
25%        9270.00000
50%       20948.50000
75%       42013.00000
max      374278.00000
Name: abs_error, dtype: float64


In [28]:
error_table.sort_values("abs_error", ascending=False)[
    ["median_income", "ocean_proximity", "actual", "predicted", "abs_error"]
].head(5)

,median_income,ocean_proximity,actual,predicted,abs_error
7150,1.8750,INLAND,500001.0,125723.0,374278.0
8042,2.3846,<1H OCEAN,500001.0,135423.0,364578.0
89,1.2434,NEAR BAY,500001.0,141733.0,358268.0
2494,3.7292,INLAND,475000.0,123980.0,351020.0
13630,1.9891,NEAR OCEAN,500000.0,151713.0,348287.0


In [29]:
error_table.groupby("ocean_proximity")["abs_error"].mean().round()

ocean_proximity
<1H OCEAN      34992.0
INLAND         23517.0
ISLAND        205178.0
NEAR BAY       39038.0
NEAR OCEAN     41887.0
Name: abs_error, dtype: float64

# Task 9.2 – Interpret Model Errors

Consider:

**1. Are the errors similar for all observations?**
**Answer:** No. They vary a lot. Using honest cross-validation predictions, half of the districts are off by less than about 21,000, but the worst are off by more than 370,000. Errors also differ by area: INLAND districts have the lowest average error (about 23,500), while the few ISLAND districts have a very large average error (about 205,000).
Note: the brief's code measures errors on the training data, giving an RMSE of about 19,300. That's too optimistic, because the forest has already seen those districts. The honest cross-validation RMSE is about 49,300.

**2. Are there observations for which the model performs particularly poorly?**
**Answer:** Yes:

- Capped districts: the worst errors are almost all districts at the 500,001 cap that have low income. The model expects low-income areas to be cheap, so it predicts around 125,000–150,000, but the recorded value is 500,000. About half of the 100 worst errors are capped districts. The cap hides the real price, which confuses the model.
- ISLAND districts: there are only a few of them in the data, so the model has almost nothing to learn from.
- Unusual districts in general: expensive areas with low income, or cheap areas with high income, don't follow the main pattern the model learned.

## Part X — Final Evaluation on the Test Set

In [30]:
final_test = strat_test_set.drop("median_house_value", axis=1)
final_test_labels = strat_test_set["median_house_value"].copy()

final_test_prepared = full_pipeline.transform(final_test)

final_predictions = final_model.predict(final_test_prepared)

final_rmse = np.sqrt(
    mean_squared_error(final_test_labels, final_predictions)
)
print("Final RMSE:", final_rmse)

Final RMSE: 49763.36683660587


In [31]:
from sklearn.metrics import mean_absolute_error

final_mae = mean_absolute_error(final_test_labels, final_predictions)
print("Final MAE:", final_mae)

Final MAE: 32476.78772609819


In [32]:
print("Test median price:", final_test_labels.median())
print("RMSE as % of median:", round(final_rmse / final_test_labels.median() * 100, 1), "%")
print("MAE as % of median:", round(final_mae / final_test_labels.median() * 100, 1), "%")

Test median price: 182150.0
RMSE as % of median: 27.3 %
MAE as % of median: 17.8 %


# Task 10.1 – Interpret the Final Result

Answer:

**1. What is the final RMSE?**
**Answer:** About 49,763 dollars.

**2. What is the final MAE?**
**Answer:** About 32,477 dollars. It's lower than the RMSE, as expected: RMSE punishes the few very large errors (like the capped districts from Part IX) more heavily, so it comes out higher.

**3. How do the final results compare with the cross-validation results?**
**Answer:** They are very close. The best grid-search model had a cross-validation RMSE of about 49,323, and the test RMSE is about 49,763. That's only about 440 worse (less than 1%). A test score slightly worse than the CV score is normal, because the model was chosen based on its CV score. This small gap shows that the cross-validation estimate was reliable, and the model is not overfitting badly. It performs on truly new data about as well as we predicted.

**4. How large is the typical prediction error relative to the house prices?**

**Answer:** The test set's median house price is about 182,150:

- RMSE ≈ 49,763 → about 27% of a typical price
- MAE ≈ 32,477 → about 18% of a typical price

So a typical prediction is off by roughly a fifth to a quarter of the house price. That's a big improvement over Linear Regression (about 38%), but still too large for decisions like setting a sale price. It's useful as a rough estimate, not an exact valuation.


## Part XI — Run a Complete Prediction Pipeline

In [33]:
from sklearn.pipeline import Pipeline

full_prediction_pipeline = Pipeline([
    ("preparation", full_pipeline),
    ("model", final_model)
])

final_predictions = full_prediction_pipeline.predict(final_test)

In [34]:
two_step = final_model.predict(full_pipeline.transform(final_test))
print("Same predictions:", np.allclose(final_predictions, two_step))

Same predictions: True


# Task 11.1 – Understand the Complete Pipeline

Explain:

**1. Which preprocessing steps are performed automatically?**
**Answer:** All the Lab 1 preparation steps, using the values already learned from the training data:

- Missing values in the numeric columns (e.g. total_bedrooms) are filled with the training median (SimpleImputer)
- Numeric columns are scaled using the training mean and standard deviation (StandardScaler)
- `ocean_proximity` is one-hot encoded into 5 columns, using the categories seen in training (OneHotEncoder)
- Then the prepared data is passed to the trained Random Forest, which makes the prediction

The pipeline only uses transform on new data, never fit, so there's no data leakage.

**2. Why is combining preprocessing and prediction useful?**

**Answer:** 

- One step instead of many: raw data in, prediction out
- Fewer mistakes: you can't forget a step, do steps in the wrong order, or accidentally use fit_transform on new data
- Consistency: new data is always prepared exactly the same way as the training data
- Easy to save and reuse: the whole system can be saved as one file and loaded later (that's Part XII), which is how models are used in real apps

## Part XII — Save and Load the Trained Model (Optional)

In [35]:
import joblib

joblib.dump(full_prediction_pipeline, "california_housing_model.pkl")

['california_housing_model.pkl']

In [36]:
loaded_model = joblib.load("california_housing_model.pkl")

predictions = loaded_model.predict(final_test)

In [37]:
print("Loaded model gives same predictions:", np.allclose(predictions, final_predictions))

Loaded model gives same predictions: True


## Part XIII — Optional Extension: Support Vector Regression

The brief says to evaluate with cross-validation but doesn't give the code. Here it is, plus a C experiment. I used cv=5 and n_jobs=-1 to keep it reasonably fast, because SVR is slow on 16,512 rows.

In [38]:
from sklearn.svm import SVR

svr_models = {
    "Linear SVR, C=1": SVR(kernel="linear", C=1),
    "Linear SVR, C=100": SVR(kernel="linear", C=100),
    "RBF SVR, C=30": SVR(kernel="rbf", C=30, gamma="scale"),
}

for name, model in svr_models.items():
    scores = cross_val_score(
        model,
        housing_prepared,
        housing_labels,
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1
    )
    rmse_scores = np.sqrt(-scores)
    print(f"{name}: Mean RMSE = {rmse_scores.mean():.0f}, Std = {rmse_scores.std():.0f}")

Linear SVR, C=1: Mean RMSE = 112590, Std = 1419
Linear SVR, C=100: Mean RMSE = 71515, Std = 825
RBF SVR, C=30: Mean RMSE = 110332, Std = 1411


# What it tells you:

- SVR with the brief's settings is very bad. About 110,000–112,000 is almost as bad as always guessing the average price (about 115,000, remember the std of the labels in Task 4.1).
- Raising C helps a lot. Linear SVR goes from about 112,590 down to about 71,515 just by changing C from 1 to 100. Same model type, very different result, just from one hyperparameter.
- Why is C=1 so bad here? My understanding (verify in your course book): house prices are in the hundreds of thousands, so a small C makes the model far too relaxed about large errors. It underfits badly. SVR usually needs much larger C values (or the target values scaled down) for data like this.
- Conclusion: even with a better C, SVR didn't beat the Random Forest here. Another reminder that hyperparameters matter, and you must compare models fairly before choosing.

# SUMMARY OF LAB 2

**A model is only as good as its score on data it has never seen.**